# Python File Handling - Solutions
### A Python Buddy Guide

---

Full solutions for all 18 questions. Each has:
- **Explanatory** - step-by-step, clear
- **Pythonic** - concise, idiomatic Python

Questions marked ★ are adapted from real OPPE exam problems.

In [ ]:
# - Setup ----------------------------------
import os, tempfile, textwrap, string

def make_file(content, suffix=".txt"):
    fd, path = tempfile.mkstemp(suffix=suffix)
    with os.fdopen(fd, 'w') as f:
        f.write(textwrap.dedent(content))
    return path

def read_file(path):
    with open(path) as f:
        return f.read()

def cleanup(*paths):
    for p in paths:
        if p and os.path.exists(p):
            os.remove(p)

print("Setup complete.")

---
## Q1 - Count Lines, Words, and Characters

In [ ]:
# --- Explanatory ---
def file_stats_v1(filepath):
    lines = words = chars = 0
    with open(filepath) as f:
        for line in f:
            lines += 1
            words += len(line.split())
            chars += len(line)
    return (lines, words, chars)

# --- Pythonic ---
def file_stats_v2(filepath):
    content = open(filepath).read()
    return (
        content.count('\n'),
        len(content.split()),
        len(content)
    )

# Tests
for fn in [file_stats_v1, file_stats_v2]:
    p = make_file("hello world\npython is fun\n")
    assert fn(p) == (2, 5, 27), f"Got {fn(p)}"
    p2 = make_file("one\n\nthree\n")
    assert fn(p2) == (3, 2, 10)
    cleanup(p, p2)
print("Q1 all passed")

**Key ideas:**
- `line.split()` with no args splits on any whitespace and strips - reliable word count.
- `len(line)` includes the `\n` - that's correct for total char count.
- Pythonic: `content.count('\n')` counts newlines = number of lines for files ending with `\n`.

---
## Q2 - Read CSV and Filter Rows

In [ ]:
# --- Explanatory ---
def filter_csv_v1(filepath, column, min_value):
    result = []
    with open(filepath) as f:
        lines = f.readlines()
    for line in lines[1:]:           # skip header
        row = line.strip().split(",")
        if int(row[column]) > min_value:
            result.append(row)
    return result

# --- Pythonic ---
def filter_csv_v2(filepath, column, min_value):
    with open(filepath) as f:
        rows = [line.strip().split(",") for line in f]
    return [r for r in rows[1:] if int(r[column]) > min_value]

# Tests
for fn in [filter_csv_v1, filter_csv_v2]:
    p = make_file("name,score,grade\nAlice,85,B\nBob,92,A\nCarol,73,C\nDave,91,A\n")
    assert fn(p, 1, 80) == [["Alice","85","B"],["Bob","92","A"],["Dave","91","A"]]
    assert fn(p, 1, 100) == []
    cleanup(p)
print("Q2 all passed")

**Key ideas:**
- `lines[1:]` skips the header - the most common CSV pattern.
- `line.strip().split(",")` handles trailing newlines before splitting.
- Compare `int(row[column])` - CSV values are always strings; cast before comparing.

---
## Q3 - Write a Numbered List to File

In [ ]:
# --- Explanatory ---
def write_numbered_v1(items, filepath):
    with open(filepath, 'w') as f:
        for i, item in enumerate(items, 1):
            f.write(f"{i}. {item}\n")
    return len(items)

# --- Pythonic ---
def write_numbered_v2(items, filepath):
    lines = [f"{i}. {item}\n" for i, item in enumerate(items, 1)]
    open(filepath, 'w').writelines(lines)
    return len(lines)

# Tests
for fn in [write_numbered_v1, write_numbered_v2]:
    fd, p = tempfile.mkstemp(); os.close(fd)
    assert fn(["apple", "banana", "cherry"], p) == 3
    assert read_file(p) == "1. apple\n2. banana\n3. cherry\n"
    assert fn([], p) == 0 and read_file(p) == ""
    cleanup(p)
print("Q3 all passed")

**Key ideas:**
- `enumerate(items, 1)` starts numbering at 1 - avoids `i+1` everywhere.
- Open in `'w'` mode to create or overwrite; use `'a'` to append.
- `f.writelines(list)` writes all strings in one call - each must include `\n`.

---
## Q4 ★ - Sum Sizes of Image Files

In [ ]:
IMG = {'jpg', 'jpeg', 'png', 'gif'}

# --- Explanatory ---
def sum_image_sizes_v1(filepath):
    total = 0
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            size, name = line.split(",", 1)
            ext = name.rsplit(".", 1)[-1].lower()
            if ext in IMG:
                total += int(size)
    return total

# --- Pythonic ---
def sum_image_sizes_v2(filepath):
    with open(filepath) as f:
        lines = [l.strip() for l in f if l.strip()]
    return sum(
        int(l.split(",",1)[0])
        for l in lines
        if l.split(",",1)[1].rsplit(".",1)[-1].lower() in IMG
    )

# Tests
for fn in [sum_image_sizes_v1, sum_image_sizes_v2]:
    p = make_file("2000,file1.jpg\n890,file2.txt\n30500,file3.JPEG\n12000,file4.png\n40000,file5.gif\n490,file6.docx\n")
    assert fn(p) == 84500
    p2 = make_file("500,img.PNG\n300,img.GIF\n")
    assert fn(p2) == 800
    cleanup(p, p2)
print("Q4 all passed")

**Key ideas:**
- `rsplit(".", 1)[-1]` splits from the right - gets the last extension even for `file.backup.jpg`.
- `.lower()` before checking the set handles mixed-case extensions like `.JPEG`.
- `split(",", 1)` limits split to 1 - safe if filename contains a comma.

---
## Q5 ★ - Reorder Jumbled Lines

In [ ]:
# --- Explanatory ---
def reorder_lines_v1(filepath):
    with open(filepath) as f:
        lines = f.readlines()
    content   = lines[:-1]                      # all but last
    order_str = lines[-1].strip()               # last line
    pairs     = [p.split("-") for p in order_str.split(",")]
    ordered   = sorted(pairs, key=lambda p: int(p[0]))
    return [content[int(orig) - 1].rstrip("\n") for _, orig in ordered]

# --- Pythonic ---
def reorder_lines_v2(filepath):
    with open(filepath) as f:
        lines = f.readlines()
    pairs = sorted(
        [p.split("-") for p in lines[-1].strip().split(",")],
        key=lambda p: int(p[0])
    )
    return [lines[int(orig) - 1].rstrip("\n") for _, orig in pairs]

# Tests
for fn in [reorder_lines_v1, reorder_lines_v2]:
    p = make_file("first line\nsecond line\nthird line\nfourth line\n1-3,2-4,3-2,4-1\n")
    assert fn(p) == ["third line", "fourth line", "second line", "first line"]
    p2 = make_file("alpha\nbeta\ngamma\n1-3,2-1,3-2\n")
    assert fn(p2) == ["gamma", "alpha", "beta"]
    cleanup(p, p2)
print("Q5 all passed")

**Key ideas:**
- `lines[-1]` is the last line (order spec); `lines[:-1]` is everything before it.
- Sort by the **new position** (`int(p[0])`), not the original position.
- Indices are 1-based → subtract 1 before indexing into the list.

---
## Q6 ★ - Fill First n Blanks Sequentially

In [ ]:
# --- Explanatory ---
def fill_blanks_v1(filepath):
    with open(filepath) as f:
        lines = f.readlines()
    n       = int(lines[0].strip())
    output  = [lines[0]]          # keep first line as-is
    counter = 1
    for line in lines[1:]:
        result = []
        for ch in line:
            if ch == '_' and counter <= n:
                result.append(str(counter))
                counter += 1
            else:
                result.append(ch)
        output.append("".join(result))
    return "".join(output)

# --- Pythonic (str.replace one at a time) ---
def fill_blanks_v2(filepath):
    with open(filepath) as f:
        lines = f.readlines()
    n, counter = int(lines[0].strip()), 1
    out = [lines[0]]
    for line in lines[1:]:
        while counter <= n and '_' in line:
            line = line.replace('_', str(counter), 1)
            counter += 1
        out.append(line)
    return "".join(out)

# Tests
for fn in [fill_blanks_v1, fill_blanks_v2]:
    p = make_file("3\na_b_c_d_\n")
    assert fn(p) == "3\na1b2c3d_\n", f"Got {repr(fn(p))}"
    p2 = make_file("4\n_x_\n_y_z\n")
    assert fn(p2) == "4\n1x2\n3y4z\n"
    cleanup(p, p2)
print("Q6 all passed")

**Key ideas:**
- Global counter spans lines - don't reset per line.
- `str.replace('_', val, 1)` replaces exactly one occurrence - safer than char-by-char for the Pythonic version.
- Output the first line unchanged; modify subsequent lines only.

---
## Q7 ★ - Rearrange String Characters by Indices

In [ ]:
# --- Explanatory ---
def rearrange_strings_v1(filepath):
    result = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts   = line.split(",")
            s       = parts[0]
            indices = [int(i) - 1 for i in parts[1:]]   # 1-based → 0-based
            result.append("".join(s[i] for i in indices))
    return result

# --- Pythonic ---
def rearrange_strings_v2(filepath):
    with open(filepath) as f:
        lines = [l.strip() for l in f if l.strip()]
    def rearrange(line):
        p = line.split(",")
        return "".join(p[0][int(i)-1] for i in p[1:])
    return [rearrange(l) for l in lines]

# Tests
for fn in [rearrange_strings_v1, rearrange_strings_v2]:
    p = make_file("hello,2,1,3,5,4\nabcdef,3,2,1,6,5,4\nxyz,3,1,2\n")
    assert fn(p) == ["ehlol", "cbafed", "zxy"]
    cleanup(p)
print("Q7 all passed")

**Key ideas:**
- Split on `","` once: `parts[0]` is the string, `parts[1:]` are the indices.
- Convert 1-based indices to 0-based by subtracting 1 before indexing.
- `"".join(s[i] for i in indices)` builds the rearranged string in one pass.

---
## Q8 ★ - Simple Stemmer

In [ ]:
# --- Explanatory ---
def stem_file_v1(filepath, suffixes):
    def stem(word):
        for sfx in suffixes:
            if word.endswith(sfx):
                return word[:-len(sfx)]
        return word

    output = []
    with open(filepath) as f:
        for line in f:
            words = line.split()
            if words:
                output.append(" ".join(stem(w) for w in words) + "\n")
    return "".join(output)

# --- Pythonic (same - logic is already clean) ---
def stem_file_v2(filepath, suffixes):
    stem = lambda w: next(
        (w[:-len(s)] for s in suffixes if w.endswith(s)), w
    )
    with open(filepath) as f:
        return "".join(
            " ".join(stem(w) for w in line.split()) + "\n"
            for line in f if line.split()
        )

# Tests
sfx = ['ations', 'ation', 'ing', 'ers', 'er', 'ly']
for fn in [stem_file_v1, stem_file_v2]:
    p = make_file("running quickly\nfaster education\n")
    assert fn(p, sfx) == "runn quick\nfast educ\n"
    p2 = make_file("nations creation\n")
    assert fn(p2, sfx) == "n cre\n"
    p3 = make_file("playing\n\nrunning\n")
    assert fn(p3, sfx) == "play\nrunn\n"
    cleanup(p, p2, p3)
print("Q8 all passed")

**Key ideas:**
- Order of suffixes matters: `'ations'` must come before `'ation'` and `'ions'`.
- `next((... for s in suffixes if word.endswith(s)), word)` - generator with default cleanly handles no-match.
- Skip blank lines: `if line.split()` is truthy only if there are words.

---
## Q9 ★ - Ubbi Dubbi

In [ ]:
VOWELS = set("aeiou")

# --- Explanatory ---
def ubbi_dubbi_v1(filepath):
    count  = 0
    output = []
    with open(filepath) as f:
        for line in f:
            result = []
            for ch in line:
                if ch in VOWELS:
                    count += 1
                    prefix = "ub" if count % 2 == 1 else "dub"
                    result.append(prefix + ch)
                else:
                    result.append(ch)
            output.append("".join(result))
    return "".join(output)

# --- Pythonic (generator expression in join) ---
def ubbi_dubbi_v2(filepath):
    count = 0
    def transform(ch):
        nonlocal count
        if ch in VOWELS:
            count += 1
            return ("ub" if count % 2 else "dub") + ch
        return ch
    with open(filepath) as f:
        return "".join(transform(ch) for line in f for ch in line)

# Tests
for fn in [ubbi_dubbi_v1, ubbi_dubbi_v2]:
    p = make_file("hello world\npython is good\n")
    assert fn(p) == "hubelldubo wuborld\npythdubon ubis gduboubod\n"
    p2 = make_file("ae\n")
    assert fn(p2) == "ubadube\n"
    cleanup(p, p2)
print("Q9 all passed")

**Key ideas:**
- The counter is **global across lines** - declare it outside the loop.
- `nonlocal count` inside a nested function allows mutation of the outer variable.
- `end=""` in print (OPPE style) is equivalent to not adding extra newlines - here we join directly.

---
## Q10 ★ - Uppercase Every k-th Vowel

In [ ]:
# --- Explanatory ---
def kth_vowel_upper_v1(filepath):
    VOWELS = set("aeiouAEIOU")
    with open(filepath) as f:
        k     = int(f.readline().strip())
        count = 0
        out   = []
        for line in f:
            result = []
            for ch in line:
                if ch.lower() in set("aeiou"):
                    count += 1
                    result.append(ch.upper() if count % k == 0 else ch.lower())
                else:
                    result.append(ch)
            out.append("".join(result))
    return "".join(out)

# --- Pythonic ---
def kth_vowel_upper_v2(filepath):
    VOWELS = set("aeiou")
    with open(filepath) as f:
        k, count = int(f.readline().strip()), 0
        def tr(ch):
            nonlocal count
            if ch.lower() in VOWELS:
                count += 1
                return ch.upper() if count % k == 0 else ch.lower()
            return ch
        return "".join(tr(ch) for line in f for ch in line)

# Tests
for fn in [kth_vowel_upper_v1, kth_vowel_upper_v2]:
    p = make_file("2\naeiou\n")
    assert fn(p) == "aEiOu\n"
    p3 = make_file("1\nhello\n")
    assert fn(p3) == "hEllO\n"
    cleanup(p, p3)
print("Q10 all passed")

**Key ideas:**
- Read `k` with `f.readline()` first - the file handle continues from line 2 onward.
- `ch.lower() in VOWELS` handles both upper and lowercase input vowels.
- `count % k == 0` identifies every k-th vowel regardless of k's value.

---
## Q11 ★ - Zigzag Character Indent

In [ ]:
# --- Explanatory ---
def zigzag_indent_v1(filepath):
    with open(filepath) as f:
        z     = int(f.readline().strip())
        cycle = 2 * (z - 1) if z > 1 else 1
        out   = []
        for i, line in enumerate(f):
            ch = line.rstrip("\n")
            if not ch:
                continue
            if z == 1:
                spaces = 0
            else:
                pos    = i % cycle
                spaces = pos if pos < z else cycle - pos
            out.append(" " * spaces + ch + "\n")
    return "".join(out)

# --- Pythonic ---
def zigzag_indent_v2(filepath):
    with open(filepath) as f:
        z = int(f.readline().strip())
        cycle = 2 * (z - 1) if z > 1 else 1
        def spaces(i):
            if z == 1: return 0
            pos = i % cycle
            return pos if pos < z else cycle - pos
        return "".join(
            " " * spaces(i) + line.rstrip("\n") + "\n"
            for i, line in enumerate(f) if line.strip()
        )

# Tests
for fn in [zigzag_indent_v1, zigzag_indent_v2]:
    p = make_file("3\na\np\np\nl\ne\n")
    assert fn(p) == "a\n p\n  p\n l\ne\n"
    p2 = make_file("1\nx\ny\nz\n")
    assert fn(p2) == "x\ny\nz\n"
    p3 = make_file("2\na\nb\nc\nd\ne\n")
    assert fn(p3) == "a\n b\nc\n d\ne\n"
    cleanup(p, p2, p3)
print("Q11 all passed")

**Key ideas:**
- Cycle length `2*(z-1)` for `z > 1`: positions 0…z-1 ascend, z…cycle-1 descend.
- `i % cycle` gives position within the current cycle.
- `pos < z` → ascending half (spaces = pos); otherwise → descending half (spaces = cycle - pos).

---
## Q12 ★ - Column Totals in a Markdown Table

In [ ]:
# --- Explanatory ---
def markdown_column_totals_v1(filepath):
    def parse_row(line):
        return [cell.strip() for cell in line.strip().strip("|").split("|")]

    with open(filepath) as f:
        lines = [l for l in f if l.strip()]

    headers   = parse_row(lines[0])            # row 0: headers
    data_rows = [parse_row(l) for l in lines[2:]]  # skip separator (row 1)

    totals = {}
    for i, header in enumerate(headers):
        totals[header] = sum(int(row[i]) for row in data_rows)
    return totals

# --- Pythonic ---
def markdown_column_totals_v2(filepath):
    parse = lambda l: [c.strip() for c in l.strip().strip("|").split("|")]
    with open(filepath) as f:
        lines = [l for l in f if l.strip()]
    headers, data = parse(lines[0]), [parse(l) for l in lines[2:]]
    return {h: sum(int(r[i]) for r in data) for i, h in enumerate(headers)}

# Tests
for fn in [markdown_column_totals_v1, markdown_column_totals_v2]:
    p = make_file("| Month | Revenue | Expenses |\n|-------|---------|----------|\n|     1 |    1000 |      600 |\n|     2 |    1500 |      800 |\n|     3 |     900 |      500 |\n")
    assert fn(p) == {"Month": 6, "Revenue": 3400, "Expenses": 1900}
    p2 = make_file("| A | B |\n|---|---|\n| 1 | 2 |\n| 3 | 4 |\n")
    assert fn(p2) == {"A": 4, "B": 6}
    cleanup(p, p2)
print("Q12 all passed")

**Key ideas:**
- `line.strip().strip("|")` removes leading/trailing `|` before splitting - handles both `| A |` and `A |`.
- `lines[1]` is always the separator row (`|---|---|`) - skip it with `lines[2:]`.
- Dict comprehension with `enumerate` maps each header to its column index cleanly.

---
## Q13 ★ - Fill Numbered Brackets

In [ ]:
# --- Explanatory (character scan) ---
def fill_brackets_v1(filepath):
    counter = 1
    out     = []
    with open(filepath) as f:
        for line in f:
            result = []
            i = 0
            while i < len(line):
                if line[i:i+2] == "[]":
                    result.append(f"[{counter}]")
                    counter += 1
                    i += 2
                else:
                    result.append(line[i])
                    i += 1
            out.append("".join(result))
    return "".join(out)

# --- Pythonic (str.replace one at a time) ---
def fill_brackets_v2(filepath):
    counter = 1
    out     = []
    with open(filepath) as f:
        for line in f:
            while "[]" in line:
                line = line.replace("[]", f"[{counter}]", 1)
                counter += 1
            out.append(line)
    return "".join(out)

# Tests
for fn in [fill_brackets_v1, fill_brackets_v2]:
    p = make_file("Programming[] is Good[]\nThe[] quick brown[] fox\njumps[] over the lazy[] dog[]\n")
    assert fn(p) == "Programming[1] is Good[2]\nThe[3] quick brown[4] fox\njumps[5] over the lazy[6] dog[7]\n"
    p2 = make_file("[x][]\n")
    assert fn(p2) == "[x][1]\n"
    cleanup(p, p2)
print("Q13 all passed")

**Key ideas:**
- Look for exactly `"[]"` (two chars) - `"[x]"` must NOT be replaced.
- `str.replace("[]", f"[{counter}]", 1)` with count=1 replaces one occurrence at a time - safe for multiple `[]` per line.
- The while loop runs until no `[]` remain in the line.

---
## Q14 - Append-Mode Structured Log Writer

In [ ]:
# --- Explanatory ---
def append_log_v1(filepath, entries):
    with open(filepath, 'a') as f:      # 'a' creates if absent, appends if exists
        for entry in entries:
            level   = entry["level"].upper()
            message = entry["message"]
            f.write(f"[{level}] {message}\n")
    with open(filepath) as f:
        return sum(1 for _ in f)        # count total lines

# --- Pythonic ---
def append_log_v2(filepath, entries):
    with open(filepath, 'a') as f:
        f.writelines(f"[{e['level'].upper()}] {e['message']}\n" for e in entries)
    return open(filepath).read().count('\n')

# Tests
for fn in [append_log_v1, append_log_v2]:
    fd, p = tempfile.mkstemp(suffix=".log"); os.close(fd); os.remove(p)
    assert fn(p, [{"level":"info","message":"started"},{"level":"error","message":"failed"}]) == 2
    assert fn(p, [{"level":"warn","message":"retrying"}]) == 3
    assert read_file(p) == "[INFO] started\n[ERROR] failed\n[WARN] retrying\n"
    cleanup(p)
print("Q14 all passed")

**Key ideas:**
- `open(path, 'a')` opens for appending - creates the file if it doesn't exist.
- `'w'` would truncate; `'a'` preserves existing content.
- Count lines by re-reading - don't track state across calls.

---
## Q15 - Merge Multiple CSV Files by Key

In [ ]:
# --- Explanatory ---
def merge_csvs_v1(filepaths, key_column):
    all_columns = []
    tables      = []

    for path in filepaths:
        with open(path) as f:
            lines = [l.strip().split(",") for l in f if l.strip()]
        header = lines[0]
        for col in header:
            if col not in all_columns:
                all_columns.append(col)
        tables.append((header, lines[1:]))

    # merged[key] = {col: value}
    merged = {}
    for header, rows in tables:
        for row in rows:
            key = row[key_column]
            if key not in merged:
                merged[key] = {}
            for col, val in zip(header, row):
                merged[key][col] = val      # later files override

    result = [all_columns]
    for key in sorted(merged):
        row = [merged[key].get(col, "") for col in all_columns]
        result.append(row)
    return result

# --- Pythonic (same algorithm, more compact) ---
merge_csvs_v2 = merge_csvs_v1   # Algorithm is the same; no meaningful shortcut

# Tests
for fn in [merge_csvs_v1]:
    p1 = make_file("id,name,score\n1,Alice,90\n2,Bob,80\n")
    p2 = make_file("id,name,grade\n2,Bobby,B\n3,Carol,A\n")
    result = fn([p1, p2], key_column=0)
    assert result[0] == ["id", "name", "score", "grade"]
    r = {row[0]: row for row in result[1:]}
    assert r["1"] == ["1", "Alice", "90", ""]
    assert r["2"] == ["2", "Bobby", "80", "B"]
    assert r["3"] == ["3", "Carol", "", "A"]
    cleanup(p1, p2)
print("Q15 all passed")

**Key ideas:**
- Build a unified column list preserving order; later files' columns are appended.
- Store merged data as `{key: {col: val}}` - later files naturally override.
- Fill missing columns with `""` when building the final rows.

---
## Q16 - Build Inverted Index from Text Files

In [ ]:
import string as _string

# --- Explanatory ---
def build_inverted_index_v1(filepaths):
    index = {}   # word → set of filepaths
    for path in filepaths:
        with open(path) as f:
            text = f.read()
        for token in text.split():
            word = token.strip(_string.punctuation).lower()
            if word:
                if word not in index:
                    index[word] = set()
                index[word].add(path)
    return {word: sorted(paths) for word, paths in index.items()}

# --- Pythonic ---
def build_inverted_index_v2(filepaths):
    index = {}
    for path in filepaths:
        words = {t.strip(_string.punctuation).lower()
                 for t in open(path).read().split() if t.strip(_string.punctuation)}
        for word in words:
            index.setdefault(word, set()).add(path)
    return {w: sorted(ps) for w, ps in index.items()}

# Tests
for fn in [build_inverted_index_v1, build_inverted_index_v2]:
    p1 = make_file("the cat sat")
    p2 = make_file("the dog ran")
    p3 = make_file("cat and dog")
    idx = fn([p1, p2, p3])
    assert sorted(idx["the"]) == sorted([p1, p2])
    assert sorted(idx["cat"]) == sorted([p1, p3])
    assert idx["sat"] == [p1]
    p4 = make_file("Hello, world! Python.")
    idx2 = fn([p4])
    assert "hello" in idx2 and "world" in idx2
    cleanup(p1, p2, p3, p4)
print("Q16 all passed")

**Key ideas:**
- `token.strip(string.punctuation)` removes leading/trailing punctuation - handles `"Hello,"` → `"hello"`.
- Use a **set** per word to automatically prevent duplicate filepath entries.
- Pythonic v2 builds a **set of unique words per file first** - avoids adding the same file to a word's index twice if the word appears multiple times.

---
## Q17 - Rotating Log File

In [ ]:
# --- Explanatory ---
def rotating_write_v1(base_path, lines, max_bytes):
    file_index  = 0
    current_path = base_path
    current_size = 0
    created      = [current_path]
    f = open(current_path, 'w')

    for line in lines:
        encoded = line + "\n"
        # Rotate if adding this line would exceed max_bytes
        # (but always write if current file is empty - handles oversized single lines)
        if current_size > 0 and current_size + len(encoded) > max_bytes:
            f.close()
            file_index  += 1
            current_path = f"{base_path}.{file_index}"
            current_size = 0
            created.append(current_path)
            f = open(current_path, 'w')
        f.write(encoded)
        current_size += len(encoded)
    f.close()
    return created

# --- Pythonic (same logic, context manager) ---
def rotating_write_v2(base_path, lines, max_bytes):
    def new_file(idx):
        path = base_path if idx == 0 else f"{base_path}.{idx}"
        return open(path, 'w'), path

    idx, size = 0, 0
    f, path = new_file(0)
    created = [path]

    for line in lines:
        data = line + "\n"
        if size > 0 and size + len(data) > max_bytes:
            f.close()
            idx += 1
            f, path = new_file(idx)
            created.append(path)
            size = 0
        f.write(data)
        size += len(data)

    f.close()
    return created

# Tests
for fn in [rotating_write_v1, rotating_write_v2]:
    base = tempfile.mktemp(suffix=".log")
    files = fn(base, ["hello", "world", "python", "rocks"], max_bytes=12)
    assert len(files) == 3
    assert read_file(files[0]) == "hello\nworld\n"
    assert read_file(files[1]) == "python\n"
    assert read_file(files[2]) == "rocks\n"
    base2 = tempfile.mktemp(suffix=".log")
    files2 = fn(base2, ["hi", "ok"], max_bytes=100)
    assert len(files2) == 1
    for f in files + files2:
        cleanup(f)
print("Q17 all passed")

**Key ideas:**
- Check `current_size > 0` before rotating - ensures a single oversized line still gets written.
- Track `current_size` in bytes (`len(encoded)`) rather than line count.
- File naming: first file is `base_path`, subsequent are `base_path.1`, `base_path.2`, …

---
## Q18 - Diff Two Files

In [ ]:
# --- Explanatory ---
def file_diff_v1(path_a, path_b):
    with open(path_a) as f:
        lines_a = [l.rstrip("\n") for l in f]
    with open(path_b) as f:
        lines_b = [l.rstrip("\n") for l in f]

    length = max(len(lines_a), len(lines_b))
    result = []
    for i in range(length):
        la = lines_a[i] if i < len(lines_a) else ""
        lb = lines_b[i] if i < len(lines_b) else ""
        if la != lb:
            result.append((i + 1, la, lb))   # 1-indexed
    return result

# --- Pythonic (itertools.zip_longest) ---
from itertools import zip_longest

def file_diff_v2(path_a, path_b):
    strip = lambda f: [l.rstrip("\n") for l in f]
    with open(path_a) as fa, open(path_b) as fb:
        la, lb = strip(fa), strip(fb)
    return [
        (i + 1, a, b)
        for i, (a, b) in enumerate(zip_longest(la, lb, fillvalue=""))
        if a != b
    ]

# Tests
for fn in [file_diff_v1, file_diff_v2]:
    pa = make_file("hello\nworld\npython\n")
    pb = make_file("hello\nearth\npython\nextra\n")
    assert fn(pa, pb) == [(2, "world", "earth"), (4, "", "extra")]
    pc = make_file("same\nlines\n")
    pd = make_file("same\nlines\n")
    assert fn(pc, pd) == []
    pe = make_file("a\nb\nc\n")
    pf = make_file("a\nb\n")
    assert fn(pe, pf) == [(3, "c", "")]
    cleanup(pa, pb, pc, pd, pe, pf)
print("Q18 all passed")

**Key ideas:**
- `rstrip("\n")` removes trailing newlines before comparison - avoids false diffs.
- `itertools.zip_longest(fillvalue="")` handles files of different lengths cleanly.
- Line numbers are 1-indexed: `i + 1` where `i` is the 0-based enumerate index.

---
## Run All Tests

In [ ]:
import traceback
from itertools import zip_longest
import string as _string

def run(name, fn):
    try:
        fn()
        print(f"PASS  {name}")
    except Exception:
        print(f"FAIL  {name}")
        traceback.print_exc()

def t01():
    p = make_file("hello world\npython is fun\n")
    assert file_stats_v2(p) == (2, 5, 27); cleanup(p)

def t02():
    p = make_file("name,score\nAlice,85\nBob,92\nCarol,73\n")
    assert filter_csv_v2(p, 1, 80) == [["Alice","85"],["Bob","92"]]; cleanup(p)

def t03():
    fd, p = tempfile.mkstemp(); os.close(fd)
    assert write_numbered_v2(["a","b"], p) == 2
    assert read_file(p) == "1. a\n2. b\n"; cleanup(p)

def t04():
    p = make_file("2000,x.jpg\n500,y.txt\n1000,z.PNG\n")
    assert sum_image_sizes_v1(p) == 3000; cleanup(p)

def t05():
    p = make_file("A\nB\nC\n1-3,2-1,3-2\n")
    assert reorder_lines_v1(p) == ["C","A","B"]; cleanup(p)

def t06():
    p = make_file("3\na_b_c_d_\n")
    assert fill_blanks_v1(p) == "3\na1b2c3d_\n"; cleanup(p)

def t07():
    p = make_file("abc,3,2,1\n")
    assert rearrange_strings_v1(p) == ["cba"]; cleanup(p)

def t08():
    sfx = ['ations','ation','ing','er','ly']
    p = make_file("running quickly\n")
    assert stem_file_v1(p, sfx) == "runn quick\n"; cleanup(p)

def t09():
    p = make_file("ae\n")
    assert ubbi_dubbi_v1(p) == "ubadube\n"; cleanup(p)

def t10():
    p = make_file("1\nhello\n")
    assert kth_vowel_upper_v1(p) == "hEllO\n"; cleanup(p)

def t11():
    p = make_file("2\na\nb\nc\n")
    assert zigzag_indent_v1(p) == "a\n b\nc\n"; cleanup(p)

def t12():
    p = make_file("| A | B |\n|---|---|\n| 1 | 2 |\n| 3 | 4 |\n")
    assert markdown_column_totals_v1(p) == {"A":4,"B":6}; cleanup(p)

def t13():
    p = make_file("[][][]\n")
    assert fill_brackets_v1(p) == "[1][2][3]\n"; cleanup(p)

def t14():
    fd, p = tempfile.mkstemp(); os.close(fd); os.remove(p)
    append_log_v1(p, [{"level":"info","message":"ok"}])
    assert read_file(p) == "[INFO] ok\n"; cleanup(p)

def t15():
    p1 = make_file("id,v\n1,a\n2,b\n")
    p2 = make_file("id,w\n2,x\n3,y\n")
    r  = merge_csvs_v1([p1,p2], 0)
    assert r[0] == ["id","v","w"]; cleanup(p1,p2)

def t16():
    p1,p2 = make_file("cat sat"),make_file("cat ran")
    idx = build_inverted_index_v1([p1,p2])
    assert sorted(idx["cat"]) == sorted([p1,p2]); cleanup(p1,p2)

def t17():
    base  = tempfile.mktemp(suffix=".log")
    files = rotating_write_v1(base,["hi","ok","bye"],max_bytes=6)
    assert len(files) == 2
    for f in files: cleanup(f)

def t18():
    pa,pb = make_file("a\nb\n"), make_file("a\nc\n")
    assert file_diff_v2(pa,pb) == [(2,"b","c")]; cleanup(pa,pb)

for i,fn in enumerate([t01,t02,t03,t04,t05,t06,t07,t08,t09,
                         t10,t11,t12,t13,t14,t15,t16,t17,t18],1):
    run(f"Q{i:02d}", fn)

print("\nAll 18 questions tested.")